In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker V3 In-Process Mode Example

This notebook demonstrates how to use SageMaker V3 ModelBuilder in In-Process mode for fast local development and testing.

In [2]:
# Import required libraries
import uuid

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.mode.function_pointers import Mode

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Step 1: Define a Simple InferenceSpec

Create a lightweight InferenceSpec for mathematical operations - perfect for in-process testing.

In [3]:
class MathInferenceSpec(InferenceSpec):
    """Simple math operations for IN_PROCESS testing."""
    
    def load(self, model_dir: str):
        """Load a simple math 'model'."""
        return {"operation": "multiply", "factor": 2.0}
    
    def invoke(self, input_object, model):
        """Perform math operation."""
        if isinstance(input_object, dict) and "numbers" in input_object:
            numbers = input_object["numbers"]
        elif isinstance(input_object, list):
            numbers = input_object
        else:
            numbers = [float(input_object)]
        
        factor = model["factor"]
        result = [num * factor for num in numbers]
        
        return {"result": result, "operation": f"multiply by {factor}"}

print("Math InferenceSpec defined successfully!")

Math InferenceSpec defined successfully!


## Step 2: Create Schema Builder

Define the expected input and output format.

In [4]:
# Create schema builder for math operations
sample_input = {"numbers": [1.0, 2.0, 3.0]}
sample_output = {"result": [2.0, 4.0, 6.0], "operation": "multiply by 2.0"}
schema_builder = SchemaBuilder(sample_input, sample_output)

print("Schema builder created successfully!")

Schema builder created successfully!


## Step 3: Configure ModelBuilder for In-Process Mode

Set up ModelBuilder to run in IN_PROCESS mode for fast local execution.

In [5]:
# Configuration
MODEL_NAME_PREFIX = "inprocess-math-model"
ENDPOINT_NAME_PREFIX = "inprocess-math-endpoint"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

# Create ModelBuilder in IN_PROCESS mode
inference_spec = MathInferenceSpec()
model_builder = ModelBuilder(
    inference_spec=inference_spec,
    schema_builder=schema_builder,
    mode=Mode.IN_PROCESS  # This is the key difference!
)

print(f"ModelBuilder configured for in-process model: {model_name}")
print(f"Target endpoint: {endpoint_name}")

[07/20/26 07:12:33] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=13726232;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=13726233;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=13726239;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=13726240;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=13726247;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=13726248;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=13726254;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=13726255;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

ModelBuilder configured for in-process model: inprocess-math-model-01667b54
Target endpoint: inprocess-math-endpoint-01667b54


## Step 4: Build the Model

Build the model - this is very fast in in-process mode!

In [6]:
# Build the model
core_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {core_model.model_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=13726262;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=13726263;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:12:35] DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=13726269;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=13726270;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#1380\1380]8;;\
                             is not handling MLflow model input                                                    

                    INFO     Cannot simulate policies for                                  ]8;id=13726275;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=13726276;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=13726281;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=13726282;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

                    INFO     ✅ Model has been created: 'inprocess-math-model-01667b54' using ]8;id=13726289;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=13726290;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#3656\3656]8;;\
                             server TORCHSERVE in IN_PROCESS mode                                                  

Model Successfully Created: inprocess-math-model-01667b54


## Step 5: Deploy Locally

Deploy the model locally - no containers or AWS resources needed!

In [7]:
# Deploy locally in in-process mode
local_endpoint = model_builder.deploy_local(endpoint_name=endpoint_name)
print(f"Local Endpoint Successfully Created: {local_endpoint.endpoint_name}")
print("Note: This runs entirely in your Python process - no containers!")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=13726295;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=13726296;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=13726301;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=13726302;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:12:36] INFO     Waiting for fastapi server to start up...                        ]8;id=13726309;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py\in_process_mode.py]8;;\:]8;id=13726310;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py#64\64]8;;\

                    WARNING  Note: This is not a standard model server.                       ]8;id=13726316;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py\in_process_mode.py]8;;\:]8;id=13726317;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py#66\66]8;;\

                    WARNING  The model is being hosted directly on the FastAPI server.        ]8;id=13726323;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py\in_process_mode.py]8;;\:]8;id=13726324;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/mode/in_process_mode.py#67\67]8;;\

...

INFO:     Application startup complete.


INFO:     Uvicorn running on http://127.0.0.1:9007 (Press CTRL+C to quit)


[07/20/26 07:12:38] INFO     Pinging local endpoint...                                       ]8;id=13726339;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/local_resources.py\local_resources.py]8;;\:]8;id=13726340;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/local_resources.py#128\128]8;;\

INFO:     127.0.0.1:43550 - "POST /invoke HTTP/1.1" 200 OK


Local Endpoint Successfully Created: inprocess-math-endpoint-01667b54
Note: This runs entirely in your Python process - no containers!


## Step 6: Test the In-Process Model

Test various mathematical operations with instant responses.

In [8]:
# Test 1: Simple multiplication
test_data_1 = {"numbers": [1.0, 2.0, 3.0]}

result_1 = local_endpoint.invoke(
    body=test_data_1,
    content_type="application/json"
)

print(f"Test 1 - Simple multiplication: {result_1.body}")

INFO:     127.0.0.1:43564 - "POST /invoke HTTP/1.1" 200 OK


Test 1 - Simple multiplication: {'result': [2.0, 4.0, 6.0], 'operation': 'multiply by 2.0'}


In [9]:
# Test 2: Larger numbers
test_data_2 = {"numbers": [10.5, 20.3, 30.7, 40.1]}

result_2 = local_endpoint.invoke(
    body=test_data_2,
    content_type="application/json"
)

print(f"Test 2 - Larger numbers: {result_2.body}")

INFO:     127.0.0.1:43574 - "POST /invoke HTTP/1.1" 200 OK


Test 2 - Larger numbers: {'result': [21.0, 40.6, 61.4, 80.2], 'operation': 'multiply by 2.0'}


In [10]:
# Test 3: Single number (alternative input format)
test_data_3 = [5.0]  # Direct list format

result_3 = local_endpoint.invoke(
    body=test_data_3,
    content_type="application/json"
)

print(f"Test 3 - Single number: {result_3.body}")

INFO:     127.0.0.1:43584 - "POST /invoke HTTP/1.1" 200 OK


Test 3 - Single number: {'result': [10.0], 'operation': 'multiply by 2.0'}


## Step 7: Performance Testing

Demonstrate the speed advantage of in-process mode.

In [11]:
import time

# Performance test - multiple rapid requests
start_time = time.time()
num_requests = 100

for i in range(num_requests):
    test_data = {"numbers": [i, i+1, i+2]}
    result = local_endpoint.invoke(
        body=test_data,
        content_type="application/json"
    )

end_time = time.time()
total_time = end_time - start_time
avg_time = total_time / num_requests

print(f"Performance Test Results:")
print(f"- Total requests: {num_requests}")
print(f"- Total time: {total_time:.3f} seconds")
print(f"- Average time per request: {avg_time*1000:.2f} ms")
print(f"- Requests per second: {num_requests/total_time:.1f}")

INFO:     127.0.0.1:43588 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:43600 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:43604 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:43610 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:43616 - "POST /invoke HTTP/1.1" 200 OK


...

INFO:     127.0.0.1:44456 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:44472 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:44478 - "POST /invoke HTTP/1.1" 200 OK


INFO:     127.0.0.1:44480 - "POST /invoke HTTP/1.1" 200 OK


Performance Test Results:
- Total requests: 100
- Total time: 0.289 seconds
- Average time per request: 2.89 ms
- Requests per second: 345.5


## Step 8: Clean Up

Clean up the in-process resources (very fast!).

In [12]:
# Clean up in-process endpoint
if local_endpoint and hasattr(local_endpoint, 'in_process_mode_obj'):
    if local_endpoint.in_process_mode_obj:
        local_endpoint.in_process_mode_obj.destroy_server()

print("In-process model and endpoint successfully cleaned up!")
print("Note: No AWS resources were created, so no cloud cleanup needed.")

[07/20/26 07:12:39] INFO     Shutting down the server...                                                 ]8;id=13726346;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_server/in_process_model_server/app.py\app.py]8;;\:]8;id=13726347;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_server/in_process_model_server/app.py#134\134]8;;\

INFO:     Shutting down


INFO:     Waiting for application shutdown.


INFO:     Application shutdown complete.


INFO:     Finished server process [70]


                    INFO     Server shutdown complete.                                                   ]8;id=13726353;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_server/in_process_model_server/app.py\app.py]8;;\:]8;id=13726354;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_server/in_process_model_server/app.py#139\139]8;;\

In-process model and endpoint successfully cleaned up!
Note: No AWS resources were created, so no cloud cleanup needed.


## Summary

This notebook demonstrated:
1. Creating a simple InferenceSpec for mathematical operations
2. Configuring ModelBuilder for IN_PROCESS mode
3. Building and deploying models locally without containers
4. Testing with various input formats
5. Performance testing showing the speed of in-process execution
6. Quick cleanup with no AWS resources

## Benefits of In-Process Mode:
- **Ultra-fast**: No container startup time
- **No AWS costs**: Runs entirely locally
- **Perfect for development**: Rapid iteration and testing
- **Easy debugging**: Direct access to Python objects
- **Lightweight**: Minimal resource usage

Use in-process mode for development, testing, and lightweight inference tasks!